In [1]:
import sys, platform, subprocess, shutil
import torch
import tensorflow as tf
import cupy as cp
from pynvml import nvmlInit, nvmlDeviceGetCount, nvmlDeviceGetHandleByIndex, nvmlDeviceGetName, nvmlDeviceGetMemoryInfo, nvmlSystemGetDriverVersion

# cell 0: quick GPU and Python environment check

print("Python:", sys.version.splitlines()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print()

# Check nvidia-smi if available
if shutil.which("nvidia-smi"):
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
            universal_newlines=True,
        ).strip()
        print("nvidia-smi output:")
        print(out)
    except Exception as e:
        print("nvidia-smi error:", e)
else:
    print("nvidia-smi: not found")
print()

# PyTorch
try:
    print("PyTorch:", torch.__version__)
    cuda_avail = torch.cuda.is_available()
    print("  CUDA available:", cuda_avail)
    if cuda_avail:
        n = torch.cuda.device_count()
        print("  CUDA devices:", n)
        for i in range(n):
            props = torch.cuda.get_device_properties(i)
            print(f"   - [{i}] {props.name}  total_memory={props.total_memory//1024**2} MB")
except Exception as e:
    print("PyTorch: not installed or error:", e)
print()

# TensorFlow
try:
    print("TensorFlow:", tf.__version__)
    try:
        gpus = tf.config.list_physical_devices("GPU")
        print("  GPUs found by TF:", len(gpus))
        for g in gpus:
            print("   -", g)
    except Exception as e:
        print("  TF GPU query error:", e)
except Exception as e:
    print("TensorFlow: not installed or error:", e)
print()

# CuPy
try:
    print("CuPy:", cp.__version__)
    try:
        cnt = cp.cuda.runtime.getDeviceCount()
        print("  CuPy visible devices:", cnt)
        for i in range(cnt):
            with cp.cuda.Device(i):
                print("   -", cp.cuda.runtime.getDeviceProperties(i)["name"].decode())
    except Exception as e:
        print("  CuPy device query error:", e)
except Exception as e:
    print("CuPy: not installed or error:", e)
print()

# pynvml (NVIDIA Management Library)
try:
    nvmlInit()
    drv = nvmlSystemGetDriverVersion()
    print("NVIDIA driver (pynvml):", drv.decode() if isinstance(drv, bytes) else drv)
    cnt = nvmlDeviceGetCount()
    print("pynvml devices:", cnt)
    for i in range(cnt):
        h = nvmlDeviceGetHandleByIndex(i)
        name = nvmlDeviceGetName(h)
        mem = nvmlDeviceGetMemoryInfo(h)
        print(f"  - [{i}] {name.decode() if isinstance(name, bytes) else name}  total={mem.total//1024**2} MB")
except Exception as e:
    print("pynvml: not installed or error:", e)

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Executable: /usr/bin/python3
Platform: Linux-6.6.113+-x86_64-with-glibc2.35

nvidia-smi output:
Tesla T4, 580.82.07, 15360 MiB

PyTorch: 2.10.0+cu128
  CUDA available: True
  CUDA devices: 1
   - [0] Tesla T4  total_memory=14912 MB

TensorFlow: 2.19.0
  GPUs found by TF: 1
   - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

CuPy: 14.0.1
  CuPy visible devices: 1
   - Tesla T4

NVIDIA driver (pynvml): 580.82.07
pynvml devices: 1
  - [0] Tesla T4  total=15360 MB
